# Online Retail Small E-Com EDA - Checkdata2

This notebook is my rough-but-structured practice file for understanding an online retail dataset.

Goal of this notebook:
- load the data properly
- check basic structure and missing values
- clean only what is needed
- understand duplicates instead of blindly deleting them
- create simple business columns like revenue
- practice groupby, filtering, correlation, and comparison questions

I am keeping notes on what command I used and why I used it, so I can come back later and revise quickly.


## 1. Loading The Dataset

What I used:
- `pd.read_excel()` to load the original Excel file
- `to_csv()` to save a CSV copy

Why I used it:
Excel files can be slower to load again and again, so saving a CSV version makes repeated practice easier.


In [ ]:
import pandas as pd
df=pd.read_excel("C:/-----SEM4-----/projects/data eng/dataset/Online Retail.xlsx")

In [ ]:
df.to_csv("C:/-----SEM4-----/projects/data eng/dataset/Online Retail.csv", index=False)
# Since Excel files take more time to load, I converted it to CSV so later practice can be faster.


In [ ]:
df.dtypes 

In [ ]:
print(df.columns)

In [ ]:
df.head()

In [ ]:
#now first let us check are there any null values in every column
print("no of null values in INVOICE NO:" , df["InvoiceNo"].isnull().sum())
print("no of null values in INVOICE Date:", df["InvoiceDate"].isnull().sum())
print("no of null values in Stockcode: " , df["StockCode"].isnull().sum())
print("no of null values in Description: " , df["Description"].isnull().sum())
print("no of null values in Quantity: " , df["Quantity"].isnull().sum())
print("no of null values in Unitprice: " , df["UnitPrice"].isnull().sum())
print("no of null values in Customer ID: " , df["CustomerID"].isnull().sum())
print("no of null values in Country: " , df["Country"].isnull().sum())


## 2. Missing Value Check

What I used:
- `.isnull().sum()` on every column

Why I used it:
Before cleaning or analysis, I need to know which columns have missing values and how serious the problem is.

What I found:
- `Description` has 1,454 missing values out of 541,909 rows, around 0.26%.
- `CustomerID` has 135,080 missing values out of 541,909 rows, around 24.9%.

So description missing values are small, but customer id missing values are a bigger issue.


## 3. Cleaning Missing Values

My thinking:
- For `Description`, I filled missing values with `no description` because the row can still be useful for quantity, price, invoice, and country analysis.
- For `CustomerID`, I dropped missing values because if customer id is not present, customer-level analysis becomes weak.

This keeps the dataset useful for customer-based questions while still preserving product/order information where possible.


In [ ]:
dfcl_null = df[df["Description"].isna() | df["CustomerID"].isna()].copy()
print(dfcl_null.shape)
dfcl_null.head()
# This shows rows where Description or CustomerID is missing.


In [ ]:
dfcl = df.copy()
dfcl["Description"] = dfcl["Description"].fillna("no description")
print("previous rows before null:", df.shape)
print("rows after filling description:", dfcl.shape)
print("description null:", dfcl["Description"].isnull().sum())
dfcl = dfcl.dropna(subset=["CustomerID"]).copy()
print("customer id null:", dfcl["CustomerID"].isnull().sum())
print("rows after dropping missing customer id:", dfcl.shape)


## 4. Duplicate Checks

What I used:
- `.duplicated().sum()` column-wise to see repeated values
- `.duplicated()` on the full dataframe to check exact duplicate rows

Why I used it:
Repeated values in a column are normal in this dataset. For example, the same customer can buy many times, and the same product can appear in many invoices.

So I should not treat column duplicates as bad data directly. Exact row duplicates need more careful checking.


In [ ]:
print("no of duplicate values in INVOICE NO:" , dfcl["InvoiceNo"].duplicated().sum())
print("no of duplicate values in INVOICE Date:", dfcl["InvoiceDate"].duplicated().sum())
print("no of duplicate values in Stockcode: " , dfcl["StockCode"].duplicated().sum())
#print("no of duplicate values in Description: " , dfcl["Description"].duplicated.sum())
print("no of duplicate values in Quantity: " , dfcl["Quantity"].duplicated().sum())
print("no of duplicate values in Unitprice: " , dfcl["UnitPrice"].duplicated().sum())
print("no of duplicate values in Customer ID: " , dfcl["CustomerID"].duplicated().sum())
print("no of duplicate values in Country: " , dfcl["Country"].duplicated().sum())


### Customer And Invoice Repetition

What I checked:
- unique customer ids
- unique invoice numbers

Why I checked it:
The duplicate count looked huge, but that is expected because customers and invoices repeat across many transaction lines. This helped me understand that duplicates are not always mistakes.


In [ ]:
print(dfcl["CustomerID"].unique().shape)
#as previously we saw duplicate value of customerid as 402457 but here there are only 4372 unique id that indicates these id people
#have been  regular customers or have been ordering more from the same online retail store
print(dfcl["InvoiceNo"].unique())

In [ ]:
#now as gpt said check wether rows are duplicated
print("exact rows duplicated are:",dfcl.duplicated().sum())
duplicates = dfcl[dfcl.duplicated(keep=False)]
duplicates.sort_values(["InvoiceNo", "StockCode", "InvoiceDate"]
).head(30)


In [ ]:
same_product = dfcl[
    (dfcl["InvoiceNo"] == 536409) &
    (dfcl["StockCode"] == 21866)
]

print(same_product)
dfcl[dfcl["InvoiceNo"] == 536409]

### Exact Duplicate Rows Decision

Exact duplicate rows were found. But identical rows can also mean the same product was entered more than once in a single invoice.

Since this dataset does not have a unique transaction-line id, I cannot confidently say every exact duplicate is an error.

Decision:
I retained them for now, because dropping them might reduce real sales quantity and revenue by mistake.


## 5. Backup Point

What I used:
- `dfcl_backup1 = dfcl.copy()`

Why I used it:
This gives me a checkpoint after initial cleaning. If I make a mistake later, I can compare or go back to this version instead of reloading everything.


In [ ]:
dfcl_backup1 = dfcl.copy()
dfcl.columns


In [ ]:
check = dfcl.loc[
    (dfcl["Description"] == "PINK CRYSTAL SKULL PHONE CHARM") |
    (dfcl["Description"] == "CREAM CUPID HEARTS COAT HANGER"),
    ["InvoiceNo", "Description", "Quantity"]
].plot(kind='bar')


In [ ]:
print(dfcl.shape)
print(df.shape)

In [ ]:
dfcl.columns

In [ ]:
uni=dfcl["StockCode"].value_counts()
uni2=dfcl["StockCode"].info
print(uni2)

## 6. Product And StockCode Exploration

What I used:
- `.value_counts()` to see common stock codes
- `.unique()` to inspect product descriptions
- filtering with `.loc[]` to compare selected products

Why I used it:
I wanted to understand how products are represented and whether `StockCode` and `Description` behave like clean product identifiers.


In [ ]:
dfcl["Description"].unique()
dfcl.columns


In [ ]:
products = [
    "WHITE METAL LANTERN",
    "CREAM CUPID HEARTS COAT HANGER"
]

cmp2= dfcl.loc[
    dfcl["Description"].isin(products),
    ["InvoiceNo", "Description"]]
print(dfcl.groupby(dfcl["Description"]=="CREAM CUPID HEARTS COAT HANGER")["UnitPrice"].mean().plot(kind='bar'))

In [ ]:
#now to understand relations between two numeric maybe use groupby
dfcl.groupby("Country")["Quantity"].describe()

In [ ]:
dfcl[["Quantity","UnitPrice"]].corr()

In [ ]:
dfcl["revenue"] = dfcl["Quantity"] * dfcl["UnitPrice"]


## 7. Practice Business Questions

Here I started asking questions from the dataset instead of only checking columns.

What I used:
- `groupby()` to create groups
- `sum()`, `mean()`, and `nunique()` to summarize values
- `sort_values()` and `head()` to find top results

Why I used it:
These commands help convert raw transaction rows into answers like top products, top customers, active countries, and revenue patterns.


In [ ]:
dfcl.groupby("Description")["revenue"].sum().sort_values(ascending=False).head(10).plot(kind='bar')


### Question: Highest Non-UK Invoice Revenue

What I used:
- filtered out United Kingdom using `.loc[]`
- grouped by `InvoiceNo` and `Country`
- summed `revenue`

Note to future me:
This code finds the biggest invoice totals outside the UK. If I want true average order value by country, I should group invoices first and then take the country average.


In [ ]:
#Which country has the highest average order value, excluding the United Kingdom?
non_uk= dfcl.loc[dfcl["Country"] != "United Kingdom"]
non_uk.groupby(["InvoiceNo","Country"])["revenue"].sum().sort_values(ascending=False).reset_index().head(10)

In [ ]:
print(dfcl.columns)
dfcl["Country"].unique()

### Question: France Customer/Product Revenue

What I used:
- filtered only France
- grouped by customer, country, and description
- summed revenue

Note to future me:
This currently finds top customer-product combinations in France. If I want top 5 customers overall, I should group only by `CustomerID` first.


In [ ]:
#Which 5 customers from France generated the highest total revenue, and how many unique products did each of those customers purchase?
only_france= dfcl.loc[dfcl["Country"]=="France"]
q2=only_france.groupby(["CustomerID","Country","Description"])["revenue"].sum().sort_values(ascending=False).head(5)
print(q2.reset_index())


### Question: Germany Products With Quantity Above 10

What I used:
- filtered Germany rows
- kept transactions where `Quantity > 10`
- grouped by product description and summed revenue

Why I used it:
This helps find products that generated strong revenue in larger-quantity German transactions.


In [ ]:
#Which 5 products generated the most revenue in Germany among customers who purchased more than 10 units in a transaction?
only_german=dfcl.loc[(dfcl["Country"]=="Germany") &(dfcl["Quantity"] > 10)]
only_german.groupby(["Description"])["revenue"].sum().sort_values(ascending = False).head(5).reset_index()

In [ ]:
nw=dfcl.groupby("Country")["CustomerID"].nunique().head(10).sort_values(ascending=False).reset_index()
print(nw)
nw.plot(kind='barh')

In [ ]:
# Which country has the highest average quantity purchased per row?
dfcl.groupby("Country")["Quantity"].mean().sort_values(ascending=False).reset_index().head(5)


In [ ]:
#now lets the the correlation in quantity and unit price
dfcl.columns

In [ ]:
dfcl[["Quantity","UnitPrice"]].corr()

## 8. Quantity Vs UnitPrice Relationship

What I used:
- `.corr()` to check correlation between `Quantity` and `UnitPrice`

What I found:
The correlation is very close to zero and slightly negative.

What it means:
There is no strong linear relationship here. So I cannot say that higher quantity clearly means higher or lower unit price from this simple correlation alone.


## 9. Country Exploration

What I used:
- `.nunique()` to count countries
- `.unique()` to inspect country names
- filtering with `.loc[]` for specific countries

Why I used it:
Country values are important for market analysis. I wanted to see which countries exist, whether any look unusual, and how much data each country has.


In [ ]:
dfcl.columns

In [ ]:
print("number of countries:", dfcl["Country"].nunique())
print(dfcl["Country"].unique())
print("cleaned dataframe shape:", dfcl.shape)


In [ ]:
only_saudiarabia = dfcl.loc[(dfcl["Country"]=="Saudi Arabia") ]
only_saudiarabia.groupby(["Country","CustomerID"])["revenue"].sum().sort_values(ascending=False).head(10).reset_index()

### Saudi Arabia Check

What I found:
Saudi Arabia has very few records compared with the full cleaned dataset.

My thinking:
This could mean the website had very low activity there, but I should not assume the reason from this dataset alone. It may be market reach, data collection, customer behavior, or just a small sample.


### Continue Country Comparison

Next idea:
Ignore very tiny country samples for a moment and compare other countries using stronger measures like total revenue, average order value, number of customers, and number of invoices.


In [ ]:
exc_saudi=dfcl.loc[dfcl["Country"] == "Saudi Arabia"]
exc_saudi.groupby(["Country","InvoiceNo","Description"])["Quantity"].sum().reset_index()

### Unspecified Country

What I noticed:
There is a country value called `Unspecified`.

My thinking:
Instead of deleting it immediately, I should investigate it. One possible check is whether the same `CustomerID` appears under a known country elsewhere. If yes, maybe some unspecified rows can be understood better.


In [ ]:
pd.crosstab(dfcl["StockCode"],dfcl["Description"],margins=True)

### StockCode And Description Mapping

What I used:
- `pd.crosstab()` between `StockCode` and `Description`

Why I used it:
I wanted to check whether one stock code always maps cleanly to one description.

This is useful because product names can be messy, renamed, or inconsistent in real datasets.


In [ ]:
nocun=dfcl.loc[dfcl["Country"] == "Unspecified"]
nocun.groupby(["Country","Description","UnitPrice"])["revenue"].nunique().reset_index()


### Handling `Unspecified` Later

Possible future approach:
- check customer ids in `Unspecified`
- see whether those same customer ids appear in other countries
- avoid guessing country unless there is clear evidence

For now, I am treating it as a data quality issue to investigate, not something to blindly remove.


In [ ]:
dfcl.describe()

## 10. What I Learned Today

I was using this notebook to understand how data works in real life: how to check quality, build relationships, compare groups, and ask better questions using pandas.

The main thing I learned is that analysis is not only about running commands. I first need to understand what the data represents, what each column means, and whether the data is clean enough for the question I am asking.

I still have a long way to go, but this notebook is now structured as a reference file. Later I can come back, see what command I used, why I used it, and improve the analysis step by step.
